In [1]:
import torch
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
import pandas as pd
from tqdm import tqdm
import json
# Load BLIP model with Vision Transformer (ViT)
device = "cuda" if torch.cuda.is_available() else "cpu"
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(device)

# Load dataset
file_path = "A:\Vishal\BiomedCLIP_data_pipeline\_results\data\pubmed_parsed_data.jsonl"
data = []
with open(file_path, "r", encoding="utf-8") as file:
    for line in file:
        data.append(json.loads(line))

df = pd.DataFrame(data)
extracted_data = []
for row in data:
    pmid = row["pmid"]
    for figure in row["figures"]:
        extracted_data.append({
            "pmid": pmid,
            "figure_id": figure["fig_id"],
            "caption": figure["fig_caption"],
            "image_path": figure["graphic_ref"]
        })

df_processed = pd.DataFrame(extracted_data)

def generate_caption(image_path):
    image = Image.open(image_path).convert("RGB")
    inputs = processor(images=image, return_tensors="pt").to(device)
    output = model.generate(**inputs)
    caption = processor.batch_decode(output, skip_special_tokens=True)[0]
    return caption

# Generating captions for dataset
df_processed["generated_caption"] = ""
for idx, row in tqdm(df_processed.iterrows(), total=len(df_processed)):
    df_processed.at[idx, "generated_caption"] = generate_caption(row["image_path"])

# Save results
df_processed.to_csv("blip_generated_captions.csv", index=False)

# Test on a single image
test_image_path = "C:/Users/admin/Downloads/chest2.jpeg"
test_caption = generate_caption(test_image_path)
print("Generated Caption:", test_caption)


a:\Vishal\myenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
a:\Vishal\myenv\lib\site-packages\huggingface_hub\file_download.py:140: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\admin\.cache\huggingface\hub\models--Salesforce--blip-image-captioning-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this art

Generated Caption: a chest with a large, white lung


In [5]:
import json
import os
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import BlipProcessor

# Define the dataset class
class CustomImageCaptionDataset(Dataset):
    def __init__(self, jsonl_file, transform=None):
        self.data = []
        with open(jsonl_file, "r", encoding="utf-8") as file:
            for line in file:
                entry = json.loads(line)
                for fig in entry["figures"]:
                    image_path = fig["graphic_ref"]
                    caption = fig["fig_caption"]
                    self.data.append((image_path, caption))

        self.transform = transform
        self.processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
        self.processor.image_processor.do_rescale = False

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        image_path, caption = self.data[idx]
        image = Image.open(image_path).convert("RGB")
        
        if self.transform:
            image = self.transform(image)
        
        inputs = self.processor(images=image, text=caption, return_tensors="pt", padding="max_length", truncation=True, max_length=64)
        return inputs["pixel_values"].squeeze(0), inputs["input_ids"].squeeze(0), inputs["attention_mask"].squeeze(0)

# Define transformations for image processing
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# Load dataset
dataset = CustomImageCaptionDataset("A:\Vishal\BiomedCLIP_data_pipeline\_results\data\pubmed_parsed_data.jsonl", transform=transform)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

# Check data
for pixel_values, input_ids, attention_mask in dataloader:
    print(pixel_values.shape, input_ids.shape, attention_mask.shape)
    break


torch.Size([4, 3, 384, 384]) torch.Size([4, 64]) torch.Size([4, 64])


In [6]:
import torch
from transformers import BlipForConditionalGeneration

# Load BLIP model
device = "cuda" if torch.cuda.is_available() else "cpu"
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(device)


In [7]:
import torch.nn as nn
import torch.optim as optim

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss(ignore_index=0)  # Ignore padding tokens
optimizer = optim.AdamW(model.parameters(), lr=5e-5)

# Training loop
num_epochs = 5

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    
    for pixel_values, input_ids, attention_mask in dataloader:
        pixel_values, input_ids, attention_mask = pixel_values.to(device), input_ids.to(device), attention_mask.to(device)
        
        optimizer.zero_grad()
        outputs = model(pixel_values=pixel_values, input_ids=input_ids, attention_mask=attention_mask, labels=input_ids)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss / len(dataloader):.4f}")


Epoch [1/5], Loss: 4.2098
Epoch [2/5], Loss: 2.9131
Epoch [3/5], Loss: 2.1867
Epoch [4/5], Loss: 1.5491
Epoch [5/5], Loss: 1.0028


In [8]:
model.save_pretrained("fine_tuned_blip")


In [3]:
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
import torch

# Load the fine-tuned model
device = "cuda" if torch.cuda.is_available() else "cpu"
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("fine_tuned_blip").to(device)

def generate_caption(image_path):
    image = Image.open(image_path).convert("RGB")
    inputs = processor(images=image, return_tensors="pt").to(device)

    # Increase max_length and use beam search
    output = model.generate(
        **inputs, 
        max_length=5000,  # Increase length of generated caption
        num_beams=5,  # Use beam search for better captions
        early_stopping=True,
        repetition_penalty=1.2  # Reduce repetitive words
    )

    caption = processor.batch_decode(output, skip_special_tokens=True)[0]
    return caption

# Example usage
image_path = "C:/Users/admin/Downloads/red_eye.jpg"
print("Generated Caption:", generate_caption(image_path))


Generated Caption: impression cytology smear stained by papanicolaou stain. this specimen was obtained from a patient with a classical dendritic ulcer clinically diagnosed as hsk. note the presence of a multinucleated giant cell ( × 500 ).


In [9]:
import os
import json

# Set the directory containing the dataset
dataset_dir = "../biomed"  # Change this to the correct path

# Get all JSON files in the dataset directory
json_files = [f for f in os.listdir(dataset_dir) if f.endswith(".json")]

# Process each JSON file
for file in json_files[:5]:  # Limiting to first 5 files for preview
    file_path = os.path.join(dataset_dir, file)
    
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

        # Extract image details
        image_file = data.get("image_file_name", "N/A")
        image_set = data.get("image_set", [])

        # Print first few image captions
        print(f"\nFile: {file}")
        print(f"Main Image: {image_file}")

        for img in image_set[:3]:  # Limiting to first 3 images per file
            print(f"  Image ID: {img.get('image_id', 'N/A')}")
            print(f"  Caption: {img.get('caption', 'No Caption Available')}")



File: filelist_commercial_batch_0_10-PMC544859-4-1743-422X-2-1-1.json
Main Image: 1743-422X-2-1-1.jpg
  Image ID: 1743-422X-2-1-4
  Caption: The effect of PI3K and MEK1/2 inhibition on RV growth and replication. Serum-starved RK13 cells were infected with RV at an m.o.i of 4 PFU/cell with or without LY294002 (30 μM) or U0126 (15 μM). Cell culture supernatants were extracted from cells at indicated time points. A – RV RNA was extracted from virus-infected cell culture supernatants and the capsid gene was amplified by RT-PCR as described under 'Experimental Procedures'. B – Monolayers of RK13 cells in 96-well plates were infected with RV-infected cell culture supernatants, and virus titers were determined using the TCID50 assay. Results are representative of a least two independent experiments.
  Image ID: 1743-422X-2-1-3
  Caption: The effect of PI3K and MEK1/2 inhibition on RV-induced apoptosis. Serum-starved RK13 cells were mock infected or infected with RV at an m.o.i of 4 PFU/cell 